# Fine-tuning a sub-1B doc-VLM on PUBLIC benchmarks — train public, validate synthetic

This is the **public-dataset** sibling of `finetune_ablation.ipynb`. The data setup is flipped:

- **Training data = real public benchmarks** — a small subset (< 200 images/benchmark) of every
  streamable dataset in the catalog, normalised into our training DTO by
  `scripts/build_benchmark_trainset.py` (see `docs/report/benchmark_trainset.md`).
- **Validation data = synthetic** — the same capability / spatial / realistic probes the other
  notebook uses, so generalisation is measured against controlled, model-free ground truth.

**Not every ablation is runnable here.** Public benchmarks give the *answer* but usually **no
spotting boxes (A1)** and **no reasoning rationale (A2)** — those arms need supervision the public
data does not carry. A feasibility cell below detects what is present and gates the arms: we run the
data-scale curve (A0), LoRA placement (A5) and preprocessing/resolution (A7), and skip A1/A2.

### Setup — sync the repo + ensure transformers>=5 (same as the synthetic notebook)

In [ ]:
# --- install docvlm_eval + ALWAYS sync to the latest pushed code + GUARANTEE transformers>=5 ---
import os, sys, subprocess, importlib
from pathlib import Path
BRANCH = "claude/new-session-w79q0i"

def _repo_root():
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "pyproject.toml").exists():
            return c
    return None

root = _repo_root()
if root is None:                       # fresh env (Colab/Kaggle): clone the repo
    subprocess.run(["git", "clone", "https://github.com/SangbumChoi/OCR.git"], check=False)
    root = Path("OCR")
root = str(root)
# A persistent runtime keeps an OLD clone -> stale scripts ("unrecognized arguments: --wandb-project").
# So ALWAYS fetch + checkout + fast-forward; if ff fails (e.g. Colab touched files), hard-reset to remote.
subprocess.run(["git", "-C", root, "fetch", "origin", BRANCH], check=False)
subprocess.run(["git", "-C", root, "checkout", BRANCH], check=False)
if subprocess.run(["git", "-C", root, "pull", "--ff-only", "origin", BRANCH]).returncode != 0:
    print(f"ff-only pull failed -> hard-resetting the clone to origin/{BRANCH}")
    subprocess.run(["git", "-C", root, "reset", "--hard", f"origin/{BRANCH}"], check=False)
subprocess.run(["git", "-C", root, "--no-pager", "log", "-1", "--oneline"], check=False)
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[newvlms,finetune,synth]"], check=True)
# belt-and-suspenders: force transformers>=5 (the <5 sweep pin / Colab's preinstalled 4.x would
# otherwise trigger "model type `qwen3_5` ... not recognized").
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5", "torchao>=0.16.0", "accelerate"], check=True)

# If a stale transformers (<5) is ALREADY imported in this kernel, the on-disk upgrade can't take
# effect until the runtime restarts (common after running the <5 sweep notebook first).
if "transformers" in sys.modules:
    import transformers as _tf
    if int(_tf.__version__.split(".")[0]) < 5:
        print(f"transformers {_tf.__version__} is loaded in this kernel; upgraded on disk -> "
              f"RESTARTING runtime. Re-run this cell after it restarts.")
        os._exit(0)                    # Colab auto-restarts the kernel

src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
import transformers, docvlm_eval
print("transformers", transformers.__version__, "| docvlm_eval", docvlm_eval.__file__)
assert int(transformers.__version__.split(".")[0]) >= 5, \
    "need transformers>=5 for the selected 2025-26 VLM (got %s)" % transformers.__version__
# sanity: confirm the freshly-synced script has the W&B flag (else the clone is still stale)
_h = subprocess.run([sys.executable, "scripts/run_ablation.py", "--help"], capture_output=True, text=True).stdout
print("run_ablation has --wandb-project:", "--wandb-project" in _h, "| --arm A0:", "A0" in _h)

### Model selection

In [ ]:
# --- MODEL SELECTION: which sub-1B base to fine-tune (everything below reads MODEL/MODELS/COLOR) ---
MODEL  = "lfm2_5-vl-1.6b"     # default: ~10x faster to train on a T4. Switch to "qwen3_5-0.8b" to use Qwen.
MODELS = [MODEL]
COLOR  = {"qwen3_5-0.8b": "#d7791d", "lfm2_5-vl-1.6b": "#1f77b4"}
HF_ID  = {"qwen3_5-0.8b": "Qwen/Qwen3.5-0.8B", "lfm2_5-vl-1.6b": "LiquidAI/LFM2.5-VL-1.6B"}
assert MODEL in HF_ID, f"MODEL must be one of {list(HF_ID)}"
print("fine-tuning base:", MODEL, "->", HF_ID[MODEL])

### Weights & Biases logging (optional)

In [ ]:
# Weights & Biases — run ONCE to log in; then every training run (A0 sizes + ablation arms) streams
# per-epoch loss and train/held-out eval metrics to your project. Skip this cell to train without W&B.
import os
os.environ["WANDB_PROJECT"] = "docvlm-ablation-public"   # edit the name; set to "" to disable logging
if os.environ["WANDB_PROJECT"]:
    try:
        import wandb
        wandb.login()                              # paste your key from https://wandb.ai/authorize
        # the A0 sweep runs run_ablation.py as a SUBPROCESS -> export the key so it authenticates too
        key = getattr(wandb.api, "api_key", None)
        if key:
            os.environ["WANDB_API_KEY"] = key
        print("W&B ready -> project:", os.environ["WANDB_PROJECT"], "| key exported to subprocess:", bool(key))
    except Exception as e:
        print("wandb login skipped:", e, "\n-> runs will train WITHOUT logging (set WANDB_PROJECT='' to silence)")

## 1. Build the public training set (once)

Streams a small subset of every catalog benchmark and writes
`data/benchmark_trainset/train.jsonl` in our `Sample` DTO. Skips the build if it already exists.
Set `--push-to-hub <repo>` (or upload the prepared `hf_dataset/` folder) to publish it once and
reload it in seconds next time.

In [ ]:
# --- build the public benchmark training set if missing, then load + summarise ---
import subprocess, sys, json
from collections import Counter
from pathlib import Path
PUBLIC_JSONL = Path.cwd() / "data" / "benchmark_trainset" / "train.jsonl"
PER_BENCH = 50   # images per benchmark (hard-capped < 200)
if not PUBLIC_JSONL.exists():
    print("building public trainset (streams ~20 HF datasets; a few minutes) ...")
    subprocess.run([sys.executable, "scripts/build_benchmark_trainset.py",
                    "--per-bench", str(PER_BENCH)], check=False)
rows = [json.loads(l) for l in PUBLIC_JSONL.read_text().splitlines() if l.strip()] \
       if PUBLIC_JSONL.exists() else []
by_b = Counter(r["meta"]["benchmark"] for r in rows)
print(f"public trainset: {len(rows)} samples over {len(by_b)} benchmarks -> {PUBLIC_JSONL}\n")
for b, c in by_b.most_common():
    print(f"  {b:14} {c}")


## 2. Feasibility gate — which ablations can public data support?

Each arm needs specific supervision. We **detect** what the loaded rows actually carry and gate:

| Arm | Needs | Public benchmarks? |
|-----|-------|--------------------|
| A0 scale curve | just (image, answer) | ✅ |
| A5 LoRA placement | nothing extra (training-side) | ✅ |
| A7 preprocessing | nothing extra (image-side) | ✅ |
| **A1 spotting** | per-answer bounding box | ❌ public VQA/OCR sets have no boxes |
| **A2 reasoning** | rationale / chain-of-thought target | ❌ public sets give the answer only |
| A4 multilingual | per-row language labels | ⚠️ trainset is English-dominant |

In [ ]:
# --- detect available supervision in the public rows and gate the arms ---
has_box = any(("box" in r["meta"]) or any(str(k).endswith("bbox") for k in r["meta"])
              or ("bbox" in str(r.get("answers"))) for r in rows)
has_rationale = any(("rationale" in r) or ("rationale" in r["meta"]) for r in rows)
multi = len({r["meta"].get("category", "") for r in rows}) > 1 and False  # no per-row lang labels
feasible = {
    "A0 learning curve (scale)":   (True,          "train public @ increasing size, validate synthetic"),
    "A5 LoRA placement":           (True,          "training-side: works on any data"),
    "A7 preprocessing/resolution": (True,          "image-side: works on any data"),
    "A1 spotting supervision":     (has_box,       "needs per-answer bbox GT -> ABSENT in public data"),
    "A2 reasoning supervision":    (has_rationale, "needs rationale/CoT GT -> ABSENT in public data"),
    "A4 multilingual mix":         (multi,         "public trainset has no per-row language labels"),
}
print(f"{'ablation':32}feasible  why")
for k, (ok, why) in feasible.items():
    print(f"{k:32}{'YES ' if ok else 'NO  '}     {why}")
print("\n-> running A0 + A5 + A7 below; A1/A2 skipped (no spotting/reasoning GT in public data).")


### Helpers — train on public, validate on synthetic (separate results file)

In [ ]:
# --- public-dataset ablation helpers (results kept separate from the synthetic notebook) ---
import os, sys, subprocess, json
import numpy as np, matplotlib.pyplot as plt
import torch
ROOT = Path.cwd()
RESULTS = ROOT / "docs" / "results" / "ablation_results_public.json"
PUB_STEPS, PUB_BATCH, PUB_MAX_IMG = 300, 2, 768
PUB_WANDB = os.environ.get("WANDB_PROJECT", "")

def _run(cmd):
    if not torch.cuda.is_available():
        print("No CUDA GPU -> NOT running. On a GPU this runs:\n  " + " ".join(cmd)); return False
    print("running:", " ".join(cmd), "\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in proc.stdout:
        print(line, end=""); buf.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 72 + "\nFAILED. last lines:\n" + "".join(buf[-25:])
              + "\n(OOM? lower PUB_BATCH/PUB_MAX_IMG.)"); return False
    return True

def run_baseline():
    """Evaluate the UNTRAINED model on the synthetic suite (the 'before')."""
    cmd = [sys.executable, "scripts/run_ablation.py", "--arm", "baseline", "--models", MODEL,
           "--results", str(RESULTS)]
    if _run(cmd): print("[ok] baseline recorded")

def run_public(record_key, placement="all", max_img=PUB_MAX_IMG, count=None, steps=PUB_STEPS):
    """LoRA-train MODEL on the public jsonl, validate on synthetic; store under record_key."""
    cmd = [sys.executable, "scripts/run_ablation.py", "--arm", "public", "--models", MODEL,
           "--train-jsonl", str(PUBLIC_JSONL), "--placement", placement, "--record-key", record_key,
           "--steps", str(steps), "--max-image-long-side", str(max_img),
           "--batch-size", str(PUB_BATCH), "--results", str(RESULTS)]
    if count: cmd += ["--count", str(count)]
    if PUB_WANDB: cmd += ["--wandb-project", PUB_WANDB]
    if _run(cmd): print(f"[ok] {record_key} recorded")

def _load(): return json.loads(RESULTS.read_text()) if RESULTS.exists() else {"models": {}}
def score(key, probe="capability", axis=None):
    r = _load().get("models", {}).get(MODEL, {}).get(key)
    if not r: return None
    s = r.get("probes", {}).get(probe)
    if not s: return None
    return s.get("by_answer_type", {}).get(axis, {}).get("score") if axis else s.get("score")
print("public results ->", RESULTS, "| MODEL =", MODEL)


## 3. A0 — learning curve: train public @ increasing scale, validate synthetic

Train the model on public subsets of growing size and score it on (a) the public **train** subset
(memorization) and (b) the **synthetic** validation set (generalization). The sizes auto-cap to the
rows available.

> **Caveat (cross-distribution).** Because train = public and validation = synthetic, the train−val
> gap reflects **domain shift on top of memorization** — it is not the pure memorization gap the
> synthetic-only A0 measures. Read the *validation* curve (does it keep rising with more public data?)
> as the transfer signal.

In [ ]:
# --- A0 public scale sweep (executes here; needs a GPU) ---
A0_SIZES = [16, 32, 64, 128, 256]   # # public images; auto-capped to what is available
A0_EPOCHS = 3
cmd = [sys.executable, "scripts/run_ablation.py", "--arm", "A0", "--models", MODEL,
       "--train-jsonl", str(PUBLIC_JSONL), "--a0-sizes", *map(str, A0_SIZES),
       "--a0-epochs", str(A0_EPOCHS), "--max-image-long-side", str(PUB_MAX_IMG),
       "--batch-size", str(PUB_BATCH), "--results", str(RESULTS)]
if PUB_WANDB: cmd += ["--wandb-project", PUB_WANDB]
_run(cmd)


In [ ]:
# --- plot A0: train (memorization) vs synthetic validation, as the public scale grows ---
d = _load().get("models", {}).get(MODEL, {}).get("A0")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
if d and d.get("sizes"):
    S = sorted(d["sizes"].values(), key=lambda s: s["n_images"])
    x = [s["n_images"] for s in S]
    tr = [s["train"].get("score") for s in S]
    va = [s["heldout"].get("score") for s in S]
    ax[0].plot(x, tr, "o-", label="train (public, memorization)")
    ax[0].plot(x, va, "s-", label="validation (synthetic, generalization)")
    ax[0].set_xlabel("# public training images"); ax[0].set_ylabel("score")
    ax[0].set_ylim(0, 1.05); ax[0].legend(); ax[0].set_title("A0 learning curve (public->synthetic)")
    gap = [(t - v) if (t is not None and v is not None) else None for t, v in zip(tr, va)]
    ax[1].plot(x, gap, "d-", color="crimson")
    ax[1].set_xlabel("# public training images"); ax[1].set_title("train - validation gap")
else:
    for a in ax: a.text(0.5, 0.5, "run the A0 cell on a GPU to populate", ha="center", va="center")
plt.tight_layout(); plt.show()


## 4. Baseline (synthetic 'before')

Evaluate the untrained model on the synthetic suite so every trained arm below has a reference.

In [ ]:
run_baseline()

## 5. A5 — LoRA placement (feasible: training-side)

Which module group to adapt — vision / connector / llm_attn / llm_mlp / all — trained on public
data, validated on the synthetic capability probe.

In [ ]:
for pl in ["vision", "connector", "llm_attn", "llm_mlp", "all"]:
    run_public(f"A5:{pl}", placement=pl)


In [ ]:
# --- plot A5 placement vs baseline (synthetic capability validation) ---
groups = ["vision", "connector", "llm_attn", "llm_mlp", "all"]
base = score("baseline", "capability")
vals = [score(f"A5:{pl}", "capability") for pl in groups]
fig, ax = plt.subplots(figsize=(8, 4))
if base is not None:
    ax.axhline(base, ls="--", color="gray", label=f"baseline = {base:.3f}")
ax.bar(groups, [v or 0 for v in vals], color="#1f77b4")
ax.set_ylim(0, 1.05); ax.set_ylabel("synthetic capability score")
ax.set_title("A5 LoRA placement — trained on public, validated on synthetic"
             + ("" if any(v is not None for v in vals) else "  (run on GPU to populate)"))
(ax.legend(fontsize=8) if ax.get_legend_handles_labels()[0] else None); plt.tight_layout(); plt.show()


## 6. A7 — preprocessing / resolution (feasible: image-side)

Vary the image long-side cap (vision-token budget). Public document images vary widely in
resolution, so this probes small-text legibility vs cost.

In [ ]:
for px in [512, 768, 1024]:
    run_public(f"A7:img{px}", placement="all", max_img=px)


In [ ]:
# --- plot A7 resolution sweep (synthetic capability validation) ---
pxs = [512, 768, 1024]
vals = [score(f"A7:img{px}", "capability") for px in pxs]
fig, ax = plt.subplots(figsize=(7, 4))
base = score("baseline", "capability")
if base is not None:
    ax.axhline(base, ls="--", color="gray", label=f"baseline = {base:.3f}")
ax.plot(pxs, [v or 0 for v in vals], "o-", color="#2a9d8f")
ax.set_xlabel("image long-side cap (px)"); ax.set_ylabel("synthetic capability score")
ax.set_ylim(0, 1.05)
ax.set_title("A7 resolution — trained on public, validated on synthetic"
             + ("" if any(v is not None for v in vals) else "  (run on GPU to populate)"))
(ax.legend(fontsize=8) if ax.get_legend_handles_labels()[0] else None); plt.tight_layout(); plt.show()


## 7. Why A1 (spotting) and A2 (reasoning) are skipped here

These arms are **not run** on public data because the supervision they require is absent:

- **A1 spotting** trains the model to emit a bounding box for the answer span. Public VQA / OCR /
  KIE benchmarks ship the *answer text* but not per-answer pixel boxes (a few detection sets have
  boxes but no question/answer), so there is no box target to learn from. The feasibility cell
  confirms `has_box = False`.
- **A2 reasoning** trains on a `rationale -> answer` target (chain-of-thought). Public sets provide
  the final answer only — no step-by-step rationale — so `has_rationale = False`.

This is exactly why the **synthetic** generator exists: it authors boxes and rationales *by
construction*, so A1/A2 are only testable in `finetune_ablation.ipynb`. The public-data notebook
covers the supervision-agnostic arms (A0 scale, A5 placement, A7 preprocessing); the synthetic
notebook covers the full set including A1/A2.

**Summary.** Train-on-public / validate-on-synthetic measures how well real-benchmark fine-tuning
*transfers* to our controlled probes, and isolates the training-side and image-side levers that do
not depend on rich annotation.